In [13]:
import pandas as pd
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.optimizer.set_jit(True)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [14]:
df = pd.read_csv("../data/processed/dia_flights.csv")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442164 entries, 0 to 442163
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Departure delay (Minutes)  442164 non-null  float64
 1   Arrival Delay (Minutes)    442164 non-null  float64
 2   temp                       442164 non-null  float64
 3   dwpt                       442164 non-null  float64
 4   rhum                       442164 non-null  float64
 5   prcp                       442164 non-null  float64
 6   wdir                       442164 non-null  float64
 7   wspd                       442164 non-null  float64
 8   pres                       442164 non-null  float64
 9   monthly_passenger_arr      442164 non-null  int64  
 10  monthly_freight_arr        442164 non-null  int64  
 11  total_arr                  442164 non-null  int64  
 12  monthly_seats_arr          442164 non-null  int64  
 13  monthly_passenger_dep      44

In [15]:
target = df["15min_delay"]
features = df.drop(columns=["15min_delay","Departure delay (Minutes)"], inplace=True)
feature_cols = df.columns
x = df[feature_cols.values]
y = target.values

#sliding window for LSTM
window = 12
x_seq = []
y_seq = []

for i in range(window,len(x)):
    x_seq.append(x[i-window:i])
    y_seq.append(y[i])

x_seq = np.array(x_seq)
y_seq = np.array(y_seq)

#split somewhat different, can't mix temporal data
split = int(0.8*len(x_seq))
x_train = x_seq[:split]
x_test = x_seq[split:]
y_train = y_seq[:split]
y_test = y_seq[split:]

model = Sequential([
    LSTM(64, return_sequences = True, input_shape=(window, x_train.shape[2])),
    Dropout(0.1),
    LSTM(32),
    Dropout(0.1),
    Dense(16, activation='relu'),
    #for binary classification vv
    Dense(1, activation='sigmoid') 
])


model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)


c:\Users\ivanl\miniconda3\envs\flight_delay_predictor\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [17]:

history = model.fit(
    x_train,y_train,
    validation_split=0.1,
    epochs=4,
    batch_size=64,
    shuffle=False,
    verbose=2
    )

Epoch 1/4
4975/4975 - 27s - 5ms/step - accuracy: 0.7514 - loss: 0.5551 - val_accuracy: 0.7129 - val_loss: 0.5998
Epoch 2/4
4975/4975 - 26s - 5ms/step - accuracy: 0.7514 - loss: 0.5552 - val_accuracy: 0.7129 - val_loss: 0.5997
Epoch 3/4
4975/4975 - 26s - 5ms/step - accuracy: 0.7514 - loss: 0.5551 - val_accuracy: 0.7129 - val_loss: 0.5997
Epoch 4/4
4975/4975 - 26s - 5ms/step - accuracy: 0.7514 - loss: 0.5551 - val_accuracy: 0.7129 - val_loss: 0.5997


In [18]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = (model.predict(x_test) >0.5).astype(int)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

2764/2764 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
              precision    recall  f1-score   support

           0       0.84      1.00      0.91     74432
           1       0.00      0.00      0.00     13999

    accuracy                           0.84     88431
   macro avg       0.42      0.50      0.46     88431
weighted avg       0.71      0.84      0.77     88431

[[74432     0]
 [13999     0]]


c:\Users\ivanl\miniconda3\envs\flight_delay_predictor\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ivanl\miniconda3\envs\flight_delay_predictor\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ivanl\miniconda3\envs\flight_delay_predictor\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [19]:
import joblib

model.save("../models/delay_clf_lstm.keras")